In [ ]:
import sys
from pathlib import Path

import grid2op
import numpy as np
from matplotlib import pyplot as plt
from lightsim2grid import LightSimBackend
import seaborn as sns

project_root = Path.cwd().parent.parent  # Adjusts for notebook being in src/visualization/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.observation_space import BusConnectivityGraphObsSpace, EDGE_INDEX
from src.visualization import visualize_graph, PlottingArgs, get_node_styles
from src.experiments.analyze_latent_graphs.build_coupling_matrices import get_PTDF_based_coupling_index

sns.reset_orig()

In [ ]:
environment = grid2op.make("l2rpn_case14_sandbox", backend=LightSimBackend())
environment.reset()
coupling_index = get_PTDF_based_coupling_index(environment)

In [ ]:
obs_space = BusConnectivityGraphObsSpace(grid2op_observation_space=environment.observation_space)
visualize_graph(PlottingArgs(
    num_nodes=57,
    node_styles=get_node_styles(environment, obs_space.__class__),
    latent_edge_probs=np.array([coupling_index, 1-coupling_index]).T,
    powerline_edge_index=obs_space.to_gym(environment.current_obs)[EDGE_INDEX]
))
plt.savefig(Path("output/ptdf_deriviation/electrical_coupling.svg"))
plt.savefig(Path("output/ptdf_deriviation/electrical_coupling.png"))
plt.show()
plt.hist(coupling_index, bins=20)
plt.xlabel("Coupling")
plt.ylabel("Frequency")
plt.savefig(Path("output/ptdf_deriviation/coupling_histogram.svg"))
plt.savefig(Path("output/ptdf_deriviation/coupling_histogram.png"))
plt.show()

In [12]:
import numpy as np
import grid2op
from lightsim2grid import LightSimBackend

env = grid2op.make("l2rpn_case14_sandbox", backend=LightSimBackend())
obs = env.reset()

grid = env.backend._grid  # lightsim2grid_cpp.GridModel
assert grid.total_bus() == 2 * env.n_sub

# Required before get_ptdf(): run one DC powerflow
Vinit = np.ones(grid.total_bus(), dtype=complex)
_ = grid.dc_pf(Vinit, 10, 1e-8)

PTDF = grid.get_ptdf()  # shape: (n_line + n_trafo, total_bus)
print(PTDF.shape)

(20, 28)
